# Meal Scan — Food Image Classifier Inference

This notebook demonstrates how to load and run inference on the trained food image classifier models. We support two lightweight export formats suitable for edge deployment:

1. **ONNX Runtime** — Ideal for backend servers or desktop environments.
2. **TensorFlow Lite (TFLite)** — Ideal for on-device mobile applications (Android/iOS).

## Model Details
- **Architecture**: EfficientNet (fine-tuned on Food-101)
- **Number of Classes**: 11 diabetic-relevant classes
- **Classes**: `baklawa`, `chourba`, `couscous`, `egg`, `french fries`, `pizza`, `sfenje`, `spaghetti`, `sushi`, `tajin_zitoun`, `tiramisu`

## 1. Setup and Preprocessing Constants

The model expects images normalized using standard ImageNet mean and standard deviation, resized to `224x224` pixels.

In [ ]:
import numpy as np
import json
from PIL import Image

# Preprocessing constants
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# Load class names
with open('./models/class_names.json', 'r') as f:
    CLASS_NAMES = json.load(f)

print(f"Loaded {len(CLASS_NAMES)} classes: {CLASS_NAMES}")

## 2. Inference using ONNX Runtime

ONNX is a high-performance inference framework. Here we load the `.onnx` model and run a prediction.

In [ ]:
import onnxruntime as ort

# Initialize ONNX Session
onnx_path = './models/food_classifier.onnx'
try:
    onnx_session = ort.InferenceSession(onnx_path)
    print("ONNX session loaded successfully.")
except Exception as e:
    print(f"Could not load ONNX model: {e}")

def predict_onnx(image_path):
    # 1. Load image and convert to RGB
    img = Image.open(image_path).convert('RGB')
    img = img.resize((224, 224))
    
    # 2. Normalize and format to CHW
    arr = np.array(img).astype(np.float32) / 255.0
    arr = (arr - np.array(MEAN, dtype=np.float32)) / np.array(STD, dtype=np.float32)
    arr = arr.transpose(2, 0, 1)[np.newaxis].astype(np.float32)  # [1, C, H, W]
    
    # 3. Run Session
    logits = onnx_session.run(None, {'input': arr})[0]
    
    # 4. Softmax
    exps = np.exp(logits - logits.max())
    probs = exps / exps.sum()
    probs = probs[0]  # remove batch dimension
    
    top_idx = probs.argmax()
    top3_idx = probs.argsort()[::-1][:3]
    
    return {
        'class': CLASS_NAMES[top_idx],
        'confidence': f"{probs[top_idx]*100:.1f}%",
        'top3': [(CLASS_NAMES[i], f"{probs[i]*100:.1f}%") for i in top3_idx]
    }

## 3. Inference using TensorFlow Lite (TFLite)

TFLite models are highly optimized for mobile devices. We can use the float32 model or the half-precision fp16 model.

In [ ]:
import tensorflow as tf

# Load TFLite Model
tflite_path = './models/food_classifier.tflite'
try:
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    
    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    print("TFLite interpreter loaded successfully.")
    print("Input Shape:", input_details[0]['shape'])
    print("Input Dtype:", input_details[0]['dtype'])
except Exception as e:
    print(f"Could not load TFLite model: {e}")

def predict_tflite(image_path):
    # 1. Load image and convert to RGB
    img = Image.open(image_path).convert('RGB').resize((224, 224))
    arr = np.array(img, dtype=np.float32) / 255.0
    arr = (arr - np.array(MEAN, dtype=np.float32)) / np.array(STD, dtype=np.float32)
    
    # 2. Match expected layout (channels-first [1, C, H, W] or channels-last [1, H, W, C])
    if input_details[0]['shape'][1] == 3:
        arr = arr.transpose(2, 0, 1)
    arr = arr[np.newaxis]
    
    # 3. Set input tensor and invoke
    interpreter.set_tensor(input_details[0]['index'], arr)
    interpreter.invoke()
    logits = interpreter.get_tensor(output_details[0]['index'])[0]
    
    # 4. Softmax
    exps = np.exp(logits - logits.max())
    probs = exps / exps.sum()
    
    top3_idx = probs.argsort()[::-1][:3]
    
    return {
        'class':      CLASS_NAMES[probs.argmax()],
        'confidence': f"{probs.max()*100:.1f}%",
        'top3': [(CLASS_NAMES[i], f"{probs[i]*100:.1f}%") for i in top3_idx]
    }

## 4. Run Example Prediction

To test the prediction, supply an image path below and run the cell.

In [ ]:
# Example Usage:
# result = predict_onnx('path_to_food_image.jpg')
# print("ONNX Result:", result)
#
# result_tflite = predict_tflite('path_to_food_image.jpg')
# print("TFLite Result:", result_tflite)
print("Uncomment and run with a valid image path to test.")